# Auditoria e exploração da base Amazon Sales

Tech Challenge Fase 4 · Gabriel da Silva Brandão Nascimento (RM374018)

Notebook de apoio ao painel em Power BI. Ele faz três coisas, nesta ordem: audita a base antes de qualquer tratamento, calcula os valores de referência para conferir as medidas do Power BI e explora a relação entre preço, desconto, engajamento e satisfação, incluindo a tabela de temas das avaliações (`produto_tema.csv`).

Para rodar, salve o `amazon.csv` (Kaggle) em `../data/` e execute as células em ordem. Algumas usam variáveis criadas em células anteriores.

In [2]:
# Configuração
import pandas as pd
 
CAMINHO_CSV = "../data/amazon.csv"  # o amazon.csv fica em data/ (baixado do Kaggle)
 
# só para as tabelas não ficarem cortadas na saída
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 200)
 
# Leio tudo como texto de propósito: se o pandas adivinhar os tipos, o "|" que está
# na coluna rating vira erro sem eu perceber. Vazios ficam como "" (keep_default_na=False).
df = pd.read_csv(CAMINHO_CSV, dtype=str, keep_default_na=False, encoding="utf-8")
 
print("Linhas:", len(df))
print("Colunas:", df.shape[1])
print(list(df.columns))

Linhas: 1465
Colunas: 16
['product_id', 'product_name', 'category', 'discounted_price', 'actual_price', 'discount_percentage', 'rating', 'rating_count', 'about_product', 'user_id', 'user_name', 'review_id', 'review_title', 'review_content', 'img_link', 'product_link']


## Granularidade e duplicatas

A base tem 1.465 linhas, mas só 1.351 `product_id` distintos, então linha não é igual a produto. Antes de tratar, quis saber se eram duplicatas de verdade ou produtos repetidos com algum campo diferente.

In [3]:
# Uma linha é um produto? Comparo o número de linhas com os product_id distintos.
ids_distintos = df["product_id"].nunique()
print("product_id distintos:", ids_distintos)
print("Linhas 'excedentes':", len(df) - ids_distintos)

product_id distintos: 1351
Linhas 'excedentes': 114


In [4]:
# Duplicata exata (as 16 colunas iguais) é uma coisa; produto repetido com conteúdo
# diferente é outra. Separo as duas.
n_exatas = df.duplicated().sum()
df_sem_exatas = df.drop_duplicates()  # cópia, o df original não muda
 
print("Duplicatas exatas:", n_exatas)
print("Linhas apos remover as exatas:", len(df_sem_exatas))
print(
    "Repeticoes com conteudo diferente:",
    len(df_sem_exatas) - df_sem_exatas["product_id"].nunique(),
)

Duplicatas exatas: 0
Linhas apos remover as exatas: 1465
Repeticoes com conteudo diferente: 114


## Qualidade das colunas

O rating tem um valor inválido: o símbolo `|` no produto B08L12N5H1. Nenhum rating está vazio, e os valores válidos vão de 2 a 5.

In [5]:
# Rating: separo vazio de texto inválido. O que não for número vira NaN.
rating_num = pd.to_numeric(df["rating"], errors="coerce")
vazio = df["rating"].str.strip() == ""
invalidos = df[rating_num.isna() & ~vazio]
 
print("Rating vazio:", vazio.sum())
print("Rating invalido (texto que nao e numero):", len(invalidos))
print(invalidos[["product_id", "rating"]])
print("Faixa dos valores validos:", rating_num.min(), "a", rating_num.max())

Rating vazio: 0
Rating invalido (texto que nao e numero): 1
      product_id rating
1279  B08L12N5H1      |
Faixa dos valores validos: 2.0 a 5.0


## O que muda nos produtos repetidos

Não há duplicata exata. O que existe são 92 produtos com mais de uma linha (206 linhas no total). Os links mudam em quase todos, o `rating_count` muda em 31 (por pouco, como a célula de materialidade mostra), e só 4 produtos têm preço divergente. As colunas de review variam em 10 a 18 produtos.

In [6]:
# Nos produtos repetidos, quais colunas mudam de uma linha para outra?
# (1 valor distinto = a coluna é constante; mais de 1 = varia)
repetidos = df_sem_exatas[df_sem_exatas.duplicated("product_id", keep=False)]
print("Linhas em ids repetidos:", len(repetidos))
print("Produtos (ids) afetados:", repetidos["product_id"].nunique())
 
distintos_por_id = repetidos.groupby("product_id").nunique()
# em quantos produtos cada coluna varia
colunas_que_variam = (distintos_por_id > 1).sum().sort_values(ascending=False)
print("\nNumero de produtos em que cada coluna VARIA:")
print(colunas_que_variam)

Linhas em ids repetidos: 206
Produtos (ids) afetados: 92

Numero de produtos em que cada coluna VARIA:
product_link           92
img_link               77
rating_count           31
review_content         18
user_id                10
review_id              10
user_name              10
review_title           10
discount_percentage     4
about_product           4
discounted_price        3
actual_price            1
rating                  0
product_name            0
category                0
dtype: int64


In [7]:
# Três exemplos de perto, só com as colunas que mudam. Serve para entender o padrão;
# quem vale é a contagem do bloco anterior.
for pid in repetidos["product_id"].drop_duplicates().head(3):
    bloco = repetidos[repetidos["product_id"] == pid]
    cols = [c for c in bloco.columns if bloco[c].nunique() > 1]
    print(f"\n=== {pid} ({len(bloco)} linhas) | colunas que variam: {cols}")
    print(bloco[cols].apply(lambda s: s.str[:70]).to_string())


=== B07JW9H4J1 (3 linhas) | colunas que variam: ['rating_count', 'review_content', 'img_link', 'product_link']
    rating_count                                                          review_content                                                                img_link                                                            product_link
0         24,269  Looks durable Charging is fine tooNo complains,Charging is really fast  https://m.media-amazon.com/images/W/WEBP_402378-T1/images/I/51UsScvHQN  https://www.amazon.in/Wayona-Braided-WN3LG1-Syncing-Charging/dp/B07JW9
369       24,270  Looks durable Charging is fine tooNo complains,Charging is really fast  https://m.media-amazon.com/images/I/51UsScvHQNL._SX300_SY300_QL70_ML2_  https://www.amazon.in/Wayona-Braided-WN3LG1-Syncing-Charging/dp/B07JW9
614       24,269  Looks durable Charging is fine tooNo complains,Charging is really fast  https://m.media-amazon.com/images/W/WEBP_402378-T2/images/I/51UsScvHQN  https://www.amazon.in/Wayon

In [8]:
# Sem os links (que mudam em quase todos), sobram quantas linhas duplicadas?
colunas_sem_links = [c for c in df.columns if c not in ("product_link", "img_link")]
print(df.duplicated(subset=colunas_sem_links).sum())

65


In [9]:
# Materialidade: a diferença no rating_count entre linhas do mesmo produto muda alguma
# conclusão? 24.269 contra 24.270 não muda nada; 100 contra 5.000 mudaria tudo.
def para_numero(serie):
    # tira ₹, % e vírgula de milhar e converte; o que não for número vira NaN
    return pd.to_numeric(serie.str.replace(r"[₹,%]", "", regex=True), errors="coerce")
 
rc = repetidos.assign(rating_count_num=para_numero(repetidos["rating_count"]))
resumo = rc.groupby("product_id")["rating_count_num"].agg(["min", "max"])
resumo["dif_abs"] = resumo["max"] - resumo["min"]
resumo["dif_pct"] = (resumo["dif_abs"] / resumo["max"] * 100).round(3)
 
varia = resumo[resumo["dif_abs"] > 0].sort_values("dif_abs", ascending=False)
print("Produtos com rating_count diferente entre as linhas:", len(varia))
print(varia.head(10))
print("\nResumo da diferenca absoluta:")
print(varia["dif_abs"].describe())

Produtos com rating_count diferente entre as linhas: 31
               min     max  dif_abs  dif_pct
product_id                                  
B0B3RRWSF6   27696   27709       13    0.047
B09YV4MW2T   29471   29478        7    0.024
B01DEWVZ2C  192587  192590        3    0.002
B0B3N7LR6K   17159   17162        3    0.017
B09MQSCJQ1   69619   69622        3    0.004
B0BDRVFDKP   67259   67262        3    0.004
B0B5B6PQCT   17831   17833        2    0.011
B09YV4RG4D   22636   22638        2    0.009
B0B3MWYCHQ   20879   20881        2    0.010
B00NH11KIK  107686  107687        1    0.001

Resumo da diferenca absoluta:
count    31.000000
mean      1.935484
std       2.393540
min       1.000000
25%       1.000000
50%       1.000000
75%       2.000000
max      13.000000
Name: dif_abs, dtype: float64


## Consistência entre os preços

Conferi se o desconto declarado bate com o recalculado a partir do preço original e do preço com desconto. Bate em todas as linhas (nenhuma diferença acima de 1 ponto) e nenhum preço com desconto é maior que o original.

In [10]:
# O desconto declarado bate com o que dá para recalcular pelos dois preços?
# O declarado vem arredondado, então aceito até 1 ponto percentual de diferença.
precos = df.assign(
    preco_desc=para_numero(df["discounted_price"]),
    preco_orig=para_numero(df["actual_price"]),
    desc_declarado=para_numero(df["discount_percentage"]),
)
precos["desc_recalculado"] = (1 - precos["preco_desc"] / precos["preco_orig"]) * 100
precos["diferenca"] = (precos["desc_declarado"] - precos["desc_recalculado"]).abs()
 
print("Preco com desconto MAIOR que o original:",
      (precos["preco_desc"] > precos["preco_orig"]).sum())
print("Linhas com diferenca > 1 ponto percentual:", (precos["diferenca"] > 1).sum())
print(precos.loc[
    precos["diferenca"] > 1,
    ["product_id", "preco_desc", "preco_orig", "desc_declarado", "desc_recalculado"],
].round(1))

Preco com desconto MAIOR que o original: 0
Linhas com diferenca > 1 ponto percentual: 0
Empty DataFrame
Columns: [product_id, preco_desc, preco_orig, desc_declarado, desc_recalculado]
Index: []


In [11]:
# Mesmo teste de duplicatas, agora ignorando maiúsculas e minúsculas
print(df.apply(lambda s: s.str.lower()).duplicated().sum())

0


## Os 4 produtos com preço divergente

Mantive a primeira ocorrência de cada produto (mesma regra do Power Query). Nos 4 casos a diferença é pequena, e o único que muda de faixa de desconto é o B096MSW6CT (90% contra 80%). O raciocínio completo está em `docs/governanca.md`.

In [12]:
# Produtos com preço diferente entre as linhas repetidas, que é a diferença que de
# fato importa. Mostro as linhas lado a lado, com o rating_count de contexto.
colunas_preco = ["discounted_price", "actual_price", "discount_percentage"]
ids_preco = distintos_por_id.index[(distintos_por_id[colunas_preco] > 1).any(axis=1)]
print("Produtos com preco divergente:", len(ids_preco))
 
for pid in ids_preco:
    bloco = repetidos[repetidos["product_id"] == pid]
    print(f"\n=== {pid} | {bloco['product_name'].iloc[0][:60]}")
    print(bloco[colunas_preco + ["rating_count"]].to_string())

Produtos com preco divergente: 4

=== B096MSW6CT | Sounce Fast Phone Charging Cable & Data Sync USB Cable Compa
    discounted_price actual_price discount_percentage rating_count
2               ₹199       ₹1,899                 90%        7,928
379             ₹199         ₹999                 80%        7,928
623             ₹199         ₹999                 80%        7,928

=== B09MT84WV5 | Samsung EVO Plus 128GB microSDXC UHS-I U3 130MB/s Full HD & 
    discounted_price actual_price discount_percentage rating_count
397           ₹1,149       ₹3,999                 71%     1,40,036
641           ₹1,059       ₹3,999                 74%     1,40,035

=== B0B5B6PQCT | boAt Wave Call Smart Watch, Smart Talk with Advanced Dedicat
    discounted_price actual_price discount_percentage rating_count
336           ₹1,999       ₹7,990                 75%       17,831
587           ₹1,799       ₹7,990                 77%       17,833

=== B0B5LVS732 | Noise Pulse Go Buzz Smart Watch Bluetooth 

## Vazios e valores fora do esperado

O único vazio são 2 células de `rating_count`. Não há preço menor ou igual a zero, rating fora de 0 a 5 nem desconto fora de 0 a 100%.

In [13]:
# Vazios e valores fora do esperado (usa a função para_numero, do bloco 7).
vazios = df.apply(lambda s: s.str.strip() == "").sum()
print("Celulas vazias por coluna (so as que tem):")
print(vazios[vazios > 0] if (vazios > 0).any() else "nenhuma")
 
num = pd.DataFrame({
    "preco_desc": para_numero(df["discounted_price"]),
    "preco_orig": para_numero(df["actual_price"]),
    "desconto": para_numero(df["discount_percentage"]),
    "rating": pd.to_numeric(df["rating"], errors="coerce"),
    "rating_count": para_numero(df["rating_count"]),
})
# count = quantos valores viraram número (o resto é NaN)
print("\nResumo numerico:")
print(num.describe().T[["count", "min", "max"]].round(1))
print("\nPrecos <= 0:", (num[["preco_desc", "preco_orig"]] <= 0).sum().to_dict())
print("Rating fora de 0-5:", ((num["rating"] < 0) | (num["rating"] > 5)).sum())
print("Desconto fora de 0-100:", ((num["desconto"] < 0) | (num["desconto"] > 100)).sum())

Celulas vazias por coluna (so as que tem):
rating_count    2
dtype: int64



Resumo numerico:
               count   min       max
preco_desc    1465.0  39.0   77990.0
preco_orig    1465.0  39.0  139900.0
desconto      1465.0   0.0      94.0
rating        1464.0   2.0       5.0
rating_count  1463.0   2.0  426973.0

Precos <= 0: {'preco_desc': 0, 'preco_orig': 0}
Rating fora de 0-5: 0
Desconto fora de 0-100: 0


## Reviews

Em 18 produtos repetidos o conteúdo de review muda entre as linhas. Decidi manter um texto por produto em vez de separar review a review (ver `docs/governanca.md`).

In [14]:
# As linhas repetidas trazem reviews diferentes ou as mesmas? O review_id guarda vários
# ids separados por vírgula, então comparo os conjuntos entre as linhas do mesmo produto:
#   em_todas == na_uniao -> mesmas reviews
#   em_todas <  na_uniao -> alguma linha tem review que as outras não têm
cols_review = ["user_id", "review_id", "review_content"]
ids_review = distintos_por_id.index[(distintos_por_id[cols_review] > 1).any(axis=1)]
print("Produtos em que as colunas de review variam:", len(ids_review))
print("Exemplo de review_id (1a linha):", repetidos["review_id"].iloc[0][:120])
 
linhas = []
for pid in ids_review:
    bloco = repetidos[repetidos["product_id"] == pid]
    conjuntos = [set(x.split(",")) for x in bloco["review_id"]]
    linhas.append({
        "product_id": pid,
        "linhas": len(bloco),
        "reviews_por_linha": [len(c) for c in conjuntos],
        "na_uniao": len(set.union(*conjuntos)),
        "em_todas": len(set.intersection(*conjuntos)),
    })
print(pd.DataFrame(linhas).to_string(index=False))

Produtos em que as colunas de review variam: 18
Exemplo de review_id (1a linha): R3HXWT0LRP0NMF,R2AJM3LFTLZHFO,R6AQJGUP6P86,R1KD19VHEDV0OR,R3C02RMYQMK6FC,R39GQRVBUZBWGY,R2K9EDOE15QIRJ,R3OI7YT648TL8I
product_id  linhas reviews_por_linha  na_uniao  em_todas
B008FWZGSG       2            [8, 8]         8         8
B077Z65HSD       3         [8, 8, 8]         8         8
B07JW9H4J1       3         [8, 8, 8]         8         8
B083342NKJ       3         [8, 8, 8]         8         8
B08BCKN299       2            [8, 8]         8         8
B08MTCKDYN       2            [8, 8]         8         8
B08WRBG3XW       2            [8, 8]         9         7
B08Y1SJVV5       2            [8, 8]         8         8
B08Y1TFSP6       3         [8, 8, 8]         8         8
B0949SBKMP       2            [8, 8]         8         8
B096MSW6CT       3         [8, 8, 8]         8         8
B097R25DP7       2            [8, 8]         9         7
B09MT84WV5       2            [8, 8]         9         7
B09

## Categorias

São 211 caminhos de categoria, com 2 a 7 níveis. O nível 1 tem 9 valores, o nível 2 tem 29 e o nível 3 tem 71. Por isso criei `categoria_n1`, `categoria_n2` e `categoria_n3` no Power Query: o nível 1 para a visão executiva, o 2 para a análise principal e o 3 para detalhe.

In [15]:
# Estrutura da coluna category. Ela não varia dentro do produto, então posso olhar
# uma linha por produto sem perder nada.
produtos = df.drop_duplicates("product_id")
niveis = produtos["category"].str.split("|", regex=False, expand=True)
 
print("Exemplos de categoria completa:")
print(produtos["category"].drop_duplicates().head(5).to_string())
 
print("\nCategorias distintas (texto completo):", produtos["category"].nunique())
print("\nQuantos NIVEIS cada produto tem (o '|' separa os niveis):")
print(produtos["category"].str.count(r"\|").add(1).value_counts().sort_index())
 
print()
for i in range(niveis.shape[1]):
    print(f"Nivel {i + 1}: {niveis[i].nunique()} valores distintos")
 
print("\nProdutos por categoria de nivel 1:")
print(niveis[0].value_counts())

Exemplos de categoria completa:
0     Computers&Accessories|Accessories&Peripherals|Cables&Accessories|C...
8     Computers&Accessories|NetworkingDevices|NetworkAdapters|WirelessUS...
12           Electronics|HomeTheater,TV&Video|Accessories|Cables|HDMICables
16            Electronics|HomeTheater,TV&Video|Televisions|SmartTelevisions
48              Electronics|HomeTheater,TV&Video|Accessories|RemoteControls

Categorias distintas (texto completo): 211

Quantos NIVEIS cada produto tem (o '|' separa os niveis):
category
2      8
3    142
4    759
5    357
6     72
7     13
Name: count, dtype: int64

Nivel 1: 9 valores distintos
Nivel 2: 29 valores distintos
Nivel 3: 71 valores distintos
Nivel 4: 136 valores distintos
Nivel 5: 56 valores distintos
Nivel 6: 14 valores distintos
Nivel 7: 7 valores distintos

Produtos por categoria de nivel 1:
0
Electronics              490
Home&Kitchen             448
Computers&Accessories    375
OfficeProducts            31
MusicalInstruments         2
Hom

## Valores de referência

Estes números servem para conferir as medidas DAX do Power BI: 1.351 produtos, 23.802.423 avaliações, nota simples 4,092 e ponderada 4,161, desconto mediano 0,49 e preço mediano 899.

In [16]:
# Valores de referência para conferir as medidas do Power BI.
# Mesma regra do Power Query: primeira ocorrência de cada product_id.
base = df.drop_duplicates("product_id").copy()  # fica a primeira ocorrência
base["preco_desc"] = para_numero(base["discounted_price"])
base["preco_orig"] = para_numero(base["actual_price"])
base["desconto"] = para_numero(base["discount_percentage"]) / 100
base["rating_n"] = pd.to_numeric(base["rating"], errors="coerce")
base["rating_count_n"] = para_numero(base["rating_count"])
 
# na média ponderada só entram produtos com nota e volume preenchidos
validos = base.dropna(subset=["rating_n", "rating_count_n"])
ponderada = (validos["rating_n"] * validos["rating_count_n"]).sum() / validos["rating_count_n"].sum()
 
print("Total de produtos:", len(base))
print("Total de avaliacoes (soma de rating_count):", int(base["rating_count_n"].sum()))
print("Nota media simples:", round(base["rating_n"].mean(), 3))
print("Nota media ponderada pelo volume:", round(ponderada, 3))
print("Desconto mediano:", round(base["desconto"].median(), 3))
print("Preco mediano com desconto:", base["preco_desc"].median())
print("Preco mediano original:", base["preco_orig"].median())

Total de produtos: 1351
Total de avaliacoes (soma de rating_count): 23802423
Nota media simples: 4.092
Nota media ponderada pelo volume: 4.161
Desconto mediano: 0.49
Preco mediano com desconto: 899.0
Preco mediano original: 1795.0


## Exploração: preço → desconto → engajamento → satisfação

A hipótese de partida era que mais desconto traria mais engajamento. Os dados não mostram isso: o Spearman deu −0,13 entre desconto e avaliações e −0,15 entre desconto e nota, ambos fracos e negativos. A faixa de 80–100% de desconto (51 produtos) tem nota ponderada de 3,99, abaixo das outras faixas (todas perto de 4,16). Aqui também saem os cortes dos percentis 25 e 75 que definem os produtos de alerta e de oportunidade.

In [17]:
# Exploração da cadeia preço → desconto → engajamento → satisfação (usa a `base` do bloco 13).
base["cat_n1"] = base["category"].str.split("|", regex=False).str[0]
base["cat_n2"] = base["category"].str.split("|", regex=False).str[1]
 
# colunas auxiliares para a nota ponderada por grupo
base["_nota_x_vol"] = base["rating_n"] * base["rating_count_n"]
base["_vol_valido"] = base["rating_count_n"].where(base["rating_n"].notna())
 
# 14.1 Spearman, que aguenta bem valores extremos (as avaliações vão de 2 a 426 mil).
# Correlação não é causa.
colunas = ["desconto", "preco_desc", "preco_orig", "rating_n", "rating_count_n"]
print("14.1 Correlacao de Spearman:")
print(base[colunas].corr(method="spearman").round(2))
 
# 14.2 Faixas de desconto
base["faixa_desconto"] = pd.cut(
    base["desconto"],
    bins=[-0.001, 0.2, 0.4, 0.6, 0.8, 1.0],
    labels=["0-20%", "20-40%", "40-60%", "60-80%", "80-100%"],
)
por_faixa = base.groupby("faixa_desconto", observed=True).agg(
    produtos=("product_id", "count"),
    nota_mediana=("rating_n", "median"),
    avaliacoes_mediana=("rating_count_n", "median"),
    preco_mediano=("preco_desc", "median"),
    nota_x_vol=("_nota_x_vol", "sum"),
    vol_valido=("_vol_valido", "sum"),
)
por_faixa["nota_ponderada"] = por_faixa["nota_x_vol"] / por_faixa["vol_valido"]
print("\n14.2 Por faixa de desconto:")
print(por_faixa.drop(columns=["nota_x_vol", "vol_valido"]).round(2))
 
# 14.3 Categorias, ignorando grupos pequenos
MIN_PRODUTOS = 20  # mínimo de produtos para o grupo entrar na leitura
 
 
def resumo_categoria(coluna):
    g = base.groupby(coluna).agg(
        produtos=("product_id", "count"),
        avaliacoes_total=("rating_count_n", "sum"),
        nota_mediana=("rating_n", "median"),
        desconto_mediano=("desconto", "median"),
        preco_mediano=("preco_desc", "median"),
        nota_x_vol=("_nota_x_vol", "sum"),
        vol_valido=("_vol_valido", "sum"),
    )
    g["nota_ponderada"] = g["nota_x_vol"] / g["vol_valido"]
    return g.drop(columns=["nota_x_vol", "vol_valido"]).sort_values("produtos", ascending=False)
 
 
print("\n14.3a Categorias de nivel 1 (todas):")
print(resumo_categoria("cat_n1").round(2))
n2 = resumo_categoria("cat_n2")
print(f"\n14.3b Categorias de nivel 2 com >= {MIN_PRODUTOS} produtos:")
print(n2[n2["produtos"] >= MIN_PRODUTOS].round(2))
 
# 14.4 Engajamento x nota, com cortes pelos percentis 25 e 75
cortes = base[["rating_n", "rating_count_n"]].quantile([0.25, 0.75])
print("\n14.4 Cortes (percentis 25 e 75):")
print(cortes)
alto_eng = base["rating_count_n"] >= cortes.loc[0.75, "rating_count_n"]
baixo_eng = base["rating_count_n"] <= cortes.loc[0.25, "rating_count_n"]
nota_baixa = base["rating_n"] <= cortes.loc[0.25, "rating_n"]
nota_alta = base["rating_n"] >= cortes.loc[0.75, "rating_n"]
print("Alto engajamento e nota baixa:", int((alto_eng & nota_baixa).sum()))
print("Nota alta e baixo engajamento:", int((nota_alta & baixo_eng).sum()))
 
colunas_ver = ["product_name", "cat_n1", "rating_n", "rating_count_n"]
print("\nAlto engajamento e nota baixa (top 10 por volume):")
print(
    base.loc[alto_eng & nota_baixa, colunas_ver]
    .assign(product_name=lambda d: d["product_name"].str[:45])
    .sort_values("rating_count_n", ascending=False)
    .head(10)
    .to_string(index=False)
)

14.1 Correlacao de Spearman:
                desconto  preco_desc  preco_orig  rating_n  rating_count_n
desconto            1.00       -0.37       -0.06     -0.15           -0.13
preco_desc         -0.37        1.00        0.93      0.08            0.15
preco_orig         -0.06        0.93        1.00      0.03            0.11
rating_n           -0.15        0.08        0.03      1.00            0.19
rating_count_n     -0.13        0.15        0.11      0.19            1.00

14.2 Por faixa de desconto:
                produtos  nota_mediana  avaliacoes_mediana  preco_mediano  nota_ponderada
faixa_desconto                                                                           
0-20%                176           4.2              6636.5          989.5            4.16
20-40%               329           4.1              7241.0         1999.0            4.17
40-60%               469           4.1              3025.0          899.0            4.17
60-80%               326           4.1    

## Será só efeito de categoria?

Para testar se a correlação geral era só categorias diferentes dando descontos diferentes, calculei de novo dentro de cada categoria de nível 2. O desconto continua andando com menos avaliações em 8 das 9 maiores (só Office Paper Products foi positiva) e com nota menor em 7 das 9 (Mobiles e Wearables foram exceções). Então não é só isso. A nota baixa da faixa de 80–100% também persiste sem os 3 maiores produtos dela (3,995).

In [18]:
# 15.1 A correlação geral pode ser só efeito de categorias diferentes darem descontos
# diferentes. Se fosse isso, ela sumiria (ou mudaria de sinal) dentro de cada categoria.
grandes = n2[n2["produtos"] >= MIN_PRODUTOS].index
linhas = []
for cat in grandes:
    g = base[base["cat_n2"] == cat]
    linhas.append({
        "cat_n2": cat,
        "produtos": len(g),
        "desc_x_avaliacoes": g["desconto"].corr(g["rating_count_n"], method="spearman"),
        "desc_x_nota": g["desconto"].corr(g["rating_n"], method="spearman"),
    })
print("15.1 Correlacao de Spearman DENTRO de cada categoria:")
print(pd.DataFrame(linhas).round(2).to_string(index=False))
 
# 15.2 Quem puxa a nota baixa da faixa 80-100%?
alto_desc = base[base["faixa_desconto"] == "80-100%"]
print("\n15.2 Faixa 80-100%: categorias (nivel 2):")
print(alto_desc["cat_n2"].value_counts().head(8))
ordenado = alto_desc.sort_values("rating_count_n", ascending=False)
print("\nMaiores produtos em volume dentro da faixa:")
print(
    ordenado[["product_name", "rating_n", "rating_count_n"]]
    .assign(product_name=lambda d: d["product_name"].str[:45])
    .head(5)
    .to_string(index=False)
)
sem_top = ordenado.iloc[3:]  # tira os 3 maiores em volume
print("Nota ponderada da faixa SEM os 3 maiores:",
      round(sem_top["_nota_x_vol"].sum() / sem_top["_vol_valido"].sum(), 3))
 
# 15.3 Concentração: % dos produtos x % das avaliações
tot_prod = len(base)
tot_aval = base["rating_count_n"].sum()
for nome, tab in [("nivel 1", resumo_categoria("cat_n1")),
                  (f"nivel 2 (>= {MIN_PRODUTOS} produtos)", n2[n2["produtos"] >= MIN_PRODUTOS])]:
    t = tab[["produtos", "avaliacoes_total", "nota_ponderada"]].copy()
    t["%produtos"] = t["produtos"] / tot_prod * 100
    t["%avaliacoes"] = t["avaliacoes_total"] / tot_aval * 100
    print(f"\n15.3 Concentracao, {nome}:")
    print(t[["produtos", "%produtos", "%avaliacoes", "nota_ponderada"]].round(1))

15.1 Correlacao de Spearman DENTRO de cada categoria:
                        cat_n2  produtos  desc_x_avaliacoes  desc_x_nota
        Kitchen&HomeAppliances       308              -0.38        -0.23
       Accessories&Peripherals       307              -0.23        -0.21
          HomeTheater,TV&Video       156              -0.41        -0.26
           Mobiles&Accessories       151              -0.38         0.15
    Heating,Cooling&AirQuality       116              -0.28        -0.09
Headphones,Earbuds&Accessories        63              -0.23        -0.30
            WearableTechnology        62               0.05         0.24
             NetworkingDevices        30              -0.24        -0.22
           OfficePaperProducts        27               0.34         0.09

15.2 Faixa 80-100%: categorias (nivel 2):
cat_n2
Accessories&Peripherals           19
Mobiles&Accessories               10
WearableTechnology                 9
Headphones,Earbuds&Accessories     6
HomeTheater,TV&Vid

## Customer Insights, passo 1: contagem de termos

Aqui comparo, em produtos de nota baixa e de nota alta, em quantos produtos cada termo aparece e o quanto ele é típico de cada grupo (lift). Serviu de ponto de partida para montar o dicionário de temas do passo seguinte.

In [19]:
# Customer Insights, passo 1: o que aparece nas avaliações de nota alta e de nota baixa?
# Conto em quantos produtos cada palavra aparece (e não quantas vezes), para um produto
# muito comentado não dominar o resultado. Usa a `base` do bloco 13.
import re
from collections import Counter
 
# Stopwords. Deixei "not" e "no" de fora da lista porque a negação muda o
# sentido ("not working", "no sound").
STOP = set("""
the and but for with about into from then once here there when where why how
all any both each few more most other some such own same than too very can will
just are was were been being does did doing have has had having
this that these those what which who whom would could should also get got
one even still really you your they them their its our his her him she
""".split())
 
 
def tokens(texto):
    palavras = re.findall(r"[a-z]+", texto.lower())
    return [p for p in palavras if len(p) >= 2 and p not in STOP]
 
 
def termos_do_produto(texto):
    t = tokens(texto)
    bigramas = {f"{a} {b}" for a, b in zip(t, t[1:])}  # pares de palavras seguidas
    return set(t) | bigramas  # SET: cada termo conta 1x por produto
 
 
cortes_nota = base["rating_n"].quantile([0.25, 0.75])
baixa = base[base["rating_n"] <= cortes_nota.loc[0.25]]
alta = base[base["rating_n"] >= cortes_nota.loc[0.75]]
print("Produtos com nota baixa:", len(baixa), "| nota alta:", len(alta))
print("Cortes de nota (p25, p75):", cortes_nota.round(2).tolist())
 
 
def contar(grupo):
    c = Counter()
    for texto in grupo["review_title"] + " " + grupo["review_content"]:
        c.update(termos_do_produto(texto))
    return c
 
 
c_baixa, c_alta = contar(baixa), contar(alta)
 
 
def mais_comuns(c, n_grupo, bigrama, k=20):
    itens = [(t, round(v / n_grupo * 100, 1)) for t, v in c.items() if (" " in t) == bigrama]
    return sorted(itens, key=lambda x: -x[1])[:k]
 
 
for nome, c, n in [("NOTA BAIXA", c_baixa, len(baixa)), ("NOTA ALTA", c_alta, len(alta))]:
    print(f"\n--- {nome}: palavras mais comuns (% dos produtos que mencionam) ---")
    print(mais_comuns(c, n, bigrama=False))
    print(f"--- {nome}: expressoes mais comuns ---")
    print(mais_comuns(c, n, bigrama=True))
 
# O que é típico de cada grupo? Razão entre as duas porcentagens (lift).
# O +0.005 evita divisão por zero e suaviza termos raros.
MIN_PROD = 10  # só termos citados por pelo menos 10 produtos do grupo
linhas = []
for t in set(c_baixa) | set(c_alta):
    pb, pa = c_baixa.get(t, 0) / len(baixa), c_alta.get(t, 0) / len(alta)
    linhas.append({
        "termo": t,
        "prod_baixa": c_baixa.get(t, 0),
        "prod_alta": c_alta.get(t, 0),
        "%baixa": round(pb * 100, 1),
        "%alta": round(pa * 100, 1),
        "lift_baixa": round((pb + 0.005) / (pa + 0.005), 2),
    })
tab = pd.DataFrame(linhas)
print("\n=== Termos mais TIPICOS de NOTA BAIXA ===")
print(tab[tab["prod_baixa"] >= MIN_PROD].sort_values("lift_baixa", ascending=False).head(25).to_string(index=False))
print("\n=== Termos mais TIPICOS de NOTA ALTA ===")
print(tab[tab["prod_alta"] >= MIN_PROD].sort_values("lift_baixa").head(25).to_string(index=False))

Produtos com nota baixa: 340 | nota alta: 419
Cortes de nota (p25, p75): [3.9, 4.3]

--- NOTA BAIXA: palavras mais comuns (% dos produtos que mencionam) ---
[('good', 97.9), ('product', 94.4), ('is', 94.4), ('it', 94.1), ('not', 89.1), ('to', 88.2), ('of', 76.2), ('in', 75.3), ('quality', 67.1), ('nice', 62.9), ('use', 57.4), ('price', 56.8), ('so', 53.2), ('money', 52.9), ('on', 51.8), ('be', 50.6), ('my', 49.7), ('as', 49.4), ('working', 47.1), ('like', 42.9)]
--- NOTA BAIXA: expressoes mais comuns ---
[('good product', 58.5), ('is good', 55.9), ('is not', 48.8), ('it is', 47.9), ('product is', 38.8), ('value money', 38.2), ('nice product', 33.8), ('quality is', 33.8), ('easy to', 31.5), ('to use', 31.2), ('it good', 27.4), ('product good', 25.0), ('good good', 22.4), ('not good', 22.1), ('good quality', 21.5), ('to be', 20.9), ('it not', 19.1), ('not working', 17.6), ('price range', 16.5), ('use it', 16.5)]

--- NOTA ALTA: palavras mais comuns (% dos produtos que mencionam) ---
[('g

## Customer Insights, passo 2: temas

Montei um dicionário de 11 temas por expressões em inglês e conferi os trechos reais impressos em 17.2 para ajustar o que dava falso positivo. Na versão final, produtos de nota baixa mencionam mais defeito (33% contra 17%) e qualidade ruim (35% contra 16%), e menos recomendação explícita (38% contra 69%). Custo-benefício quase não separa os grupos (51% contra 56%).

O método detecta menção de palavras, não sentimento. A saída é o `produto_tema.csv`, que o Power BI usa.

In [20]:
# Customer Insights, passo 2: temas por dicionário de expressões, nota baixa x nota alta.
# As reviews estão em inglês, então as expressões também. O dicionário é uma hipótese:
# conferi lendo os trechos de 17.2 e ajustei o que dava falso positivo.
# tipo: Problema (linguagem negativa), Ponto forte (positiva) ou Tópico (assunto).
A = r"[’']"  # aceita os dois tipos de apóstrofo
TEMAS = {
    "Defeito / nao funciona": ("Problema",
        rf"not working|stopped working|doesn{A}?t work|don{A}?t work|not work|defective|faulty|dead on arrival|malfunction|not charging|stopped charging"),
    "Qualidade ruim / insatisfacao": ("Problema",
        r"poor quality|low quality|bad quality|cheap quality|worst|waste of money|not worth|not durable|not satisfied|disappoint|broke|broken"),
    "Alerta a compradores": ("Problema",
        rf"(?:don{A}?t|do not|never) buy (?:this|it)|(?:not|never|don{A}?t) recommend|(?:don{A}?t|do not) waste your money"),
    "Audio (grave, ruido, microfone)": ("Topico",
        r"bass|noise cancel|microphone|mic quality|call quality|sound leak"),
    "Bateria / autonomia": ("Topico",
        r"battery life|battery backup|battery drain|drains"),
    "Garantia / troca / reembolso": ("Topico",
        r"replacement|refund|warranty|returned|return policy|customer service|customer care|service cent"),
    "Entrega / embalagem": ("Topico",
        r"\bdeliver|\bpackag|\bpacked\b|\barrived\b"),
    "Custo-beneficio": ("Ponto forte",
        r"value for money|worth the|worth every|affordable|reasonable price|good price|price range"),
    "Qualidade / durabilidade": ("Ponto forte",
        r"good quality|great quality|excellent quality|sturdy|durable|long lasting|well built|build quality"),
    "Facilidade de uso": ("Ponto forte",
        r"easy to (?:use|install|setup|set up|carry|clean|operate|handle)|user friendly"),
    "Recomendacao explicita": ("Ponto forte",
        r"highly recommend|must buy|go for it|love it|awesome|superb|excellent|worth buying"),
}
 
base["texto"] = (base["review_title"] + " " + base["review_content"]).str.lower()
# Nos pontos fortes tiro a negação e as 3 palavras seguintes
# ("not quite durable" não pode contar como elogio).
NEG = re.compile(
    rf"\b(?:not|never|no|cannot|can{A}t|isn{A}t|wasn{A}t|doesn{A}t|didn{A}t|don{A}t|hardly)\b(?:\s+\w+){{0,3}}"
)
base["texto_sem_neg"] = base["texto"].apply(lambda s: NEG.sub(" ", s))
 
mask_baixa = base["rating_n"] <= cortes_nota.loc[0.25]
mask_alta = base["rating_n"] >= cortes_nota.loc[0.75]
 
linhas = []
flags = {}
for tema, (tipo, padrao) in TEMAS.items():
    coluna_texto = "texto_sem_neg" if tipo == "Ponto forte" else "texto"
    flag = base[coluna_texto].str.contains(padrao, regex=True)
    flags[tema] = flag
    linhas.append({
        "tema": tema, "tipo": tipo, "produtos": int(flag.sum()),
        "%todos": round(flag.mean() * 100, 1),
        "%nota_baixa": round(flag[mask_baixa].mean() * 100, 1),
        "%nota_alta": round(flag[mask_alta].mean() * 100, 1),
    })
print("17.1 Temas: % dos produtos que mencionam")
print(pd.DataFrame(linhas).to_string(index=False))
 
# 17.2 Validação: 3 trechos reais por tema, para ver se a expressão faz sentido
print("\n17.2 Trechos de exemplo (validacao do dicionario):")
for tema, (tipo, padrao) in TEMAS.items():
    coluna_texto = "texto_sem_neg" if tipo == "Ponto forte" else "texto"
    print(f"\n[{tema}]")
    for texto in base.loc[flags[tema], coluna_texto].head(3):
        m = re.search(padrao, texto)
        ini, fim = max(0, m.start() - 50), min(len(texto), m.end() + 50)
        print("   ...", texto[ini:fim].replace("\n", " "), "...")
 
# 17.3 Exporta a tabela (1 linha por produto x tema) para o Power BI
longa = [
    (pid, tema, TEMAS[tema][0])
    for tema in TEMAS
    for pid in base.loc[flags[tema], "product_id"]
]
saida = pd.DataFrame(longa, columns=["product_id", "tema", "tipo"])
saida.to_csv("produto_tema.csv", index=False, encoding="utf-8-sig")
print(f"\n17.3 Gerado produto_tema.csv na pasta atual: {len(saida)} linhas")
 
# 17.4 Fones: o que diferencia os de nota <= 3,9 dos demais?
FONES = "Headphones,Earbuds&Accessories"
fones = base["cat_n2"] == FONES
if fones.sum() == 0:
    print("\n17.4 Categoria de fones nao encontrada")
else:
    f_baixa = fones & (base["rating_n"] <= 3.9)
    f_resto = fones & (base["rating_n"] > 3.9)
    print(f"\n17.4 Fones: {int(f_baixa.sum())} produtos com nota <= 3,9 | {int(f_resto.sum())} com nota > 3,9")
    zoom = pd.DataFrame({
        "tipo": {t: v[0] for t, v in TEMAS.items()},
        "%fones_nota_baixa": {t: round(flags[t][f_baixa].mean() * 100, 1) for t in TEMAS},
        "%fones_nota_maior": {t: round(flags[t][f_resto].mean() * 100, 1) for t in TEMAS},
    })
    print(zoom.to_string())

17.1 Temas: % dos produtos que mencionam
                           tema        tipo  produtos  %todos  %nota_baixa  %nota_alta
         Defeito / nao funciona    Problema       327    24.2         32.6        16.7
  Qualidade ruim / insatisfacao    Problema       312    23.1         34.7        16.0
           Alerta a compradores    Problema        79     5.8         10.6         3.3
Audio (grave, ruido, microfone)      Topico       111     8.2         12.9         3.1
            Bateria / autonomia      Topico       172    12.7         13.5         9.1
   Garantia / troca / reembolso      Topico       301    22.3         21.2        19.3
            Entrega / embalagem      Topico       386    28.6         22.6        35.8
                Custo-beneficio Ponto forte       766    56.7         50.9        56.1
       Qualidade / durabilidade Ponto forte       660    48.9         43.5        54.4
              Facilidade de uso Ponto forte       434    32.1         28.8        36.8
  

## Evidência para as recomendações

Os números usados nas recomendações do painel e do relatório (onde estão os 49 produtos em alerta e os 81 de oportunidade, e quanto cada categoria recebe de avaliações) saem desta célula. Os cortes são os mesmos do Power BI: percentis 25 e 75 de avaliações e notas de 3,9 e 4,3. Entre os 81 de oportunidade, 33 (40,7%) estão em Home & Kitchen, onde o produto típico recebe 2.306 avaliações, contra 9.130 em Electronics (medianas). Entre os 49 em alerta, 30 são de Electronics e 14 são fones.

In [21]:
# Evidência dos números do painel: produtos em alerta e de oportunidade, onde estão e o
# quanto cada categoria recebe de avaliações. Usa a `base` (um registro por produto).
# Os cortes são os mesmos do Power BI: percentis 25 e 75 de avaliações, notas 3,9 e 4,3.
p75_aval = base["rating_count_n"].quantile(0.75)
p25_aval = base["rating_count_n"].quantile(0.25)
alerta = base[(base["rating_count_n"] >= p75_aval) & (base["rating_n"] <= 3.9)]
oport = base[(base["rating_count_n"] <= p25_aval) & (base["rating_n"] >= 4.3)]
print(f"Cortes de avaliações: P75 = {p75_aval:,.0f} | P25 = {p25_aval:,.0f}")
print(f"Produtos em alerta: {len(alerta)} | de oportunidade: {len(oport)}")

print("\nAlerta por categoria (nível 1):")
print(alerta["cat_n1"].value_counts().to_string())
print("\nAlerta por categoria (nível 2), maiores:")
print(alerta["cat_n2"].value_counts().head(4).to_string())

print("\nOportunidade por categoria (nível 1):")
vc = oport["cat_n1"].value_counts()
print(pd.DataFrame({"produtos": vc, "%": (vc / len(oport) * 100).round(1)}).to_string())

# Quanto o produto típico de cada categoria recebe de avaliações
print("\nAvaliações por produto (nível 1):")
comp = base[base["cat_n1"].isin(["Electronics", "Home&Kitchen"])].groupby("cat_n1")["rating_count_n"]
print(comp.agg(mediana="median", media="mean").round(0).astype(int).to_string())

print("\nMediana de preço e de desconto:")
print(pd.DataFrame({
    "preco_mediano": {"alerta": alerta["preco_desc"].median(), "oportunidade": oport["preco_desc"].median(), "todos": base["preco_desc"].median()},
    "desconto_mediano_%": {"alerta": alerta["desconto"].median() * 100, "oportunidade": oport["desconto"].median() * 100, "todos": base["desconto"].median() * 100},
}).round(0).to_string())

Cortes de avaliações: P75 = 16,020 | P25 = 1,106
Produtos em alerta: 49 | de oportunidade: 81

Alerta por categoria (nível 1):
cat_n1
Electronics              30
Home&Kitchen             12
Computers&Accessories     6
MusicalInstruments        1

Alerta por categoria (nível 2), maiores:
cat_n2
Headphones,Earbuds&Accessories    14
Kitchen&HomeAppliances            12
WearableTechnology                 7
Mobiles&Accessories                6

Oportunidade por categoria (nível 1):
                       produtos     %
cat_n1                               
Home&Kitchen                 33  40.7
Electronics                  26  32.1
Computers&Accessories        21  25.9
OfficeProducts                1   1.2

Avaliações por produto (nível 1):
              mediana  media
cat_n1                      
Electronics      9130  28997
Home&Kitchen     2306   6676

Mediana de preço e de desconto:
              preco_mediano  desconto_mediano_%
alerta                899.0                60.0
oportunida

Ressalva sobre os 81 produtos de oportunidade: 21 deles têm menos de 100 avaliações (o menor tem 5; a mediana é 356). Com tão poucas avaliações a nota ainda é instável, por isso a recomendação é começar o destaque como teste. Nos 49 em alerta a nota vai de 3,5 a 3,9, abaixo da média ponderada do catálogo (4,16), mas não é insatisfação extrema.

In [22]:
# Robustez da nota nos 81 produtos de oportunidade: nota de poucas avaliações é instável
# (cinco estrelas dadas por cinco pessoas ainda não provam qualidade).
print("Avaliações por produto de oportunidade:")
print(oport["rating_count_n"].describe().round(0).to_string())
for corte in (50, 100, 200):
    print(f"Com menos de {corte} avaliações: {int((oport['rating_count_n'] < corte).sum())} de {len(oport)}")
print("\nProdutos em alerta: nota de", alerta["rating_n"].min(), "a", alerta["rating_n"].max(),
      "| média", round(alerta["rating_n"].mean(), 2), "| catálogo (ponderada):",
      round(base["_nota_x_vol"].sum() / base["_vol_valido"].sum(), 2))

Avaliações por produto de oportunidade:
count      81.0
mean      388.0
std       316.0
min         5.0
25%        97.0
50%       356.0
75%       592.0
max      1065.0
Com menos de 50 avaliações: 12 de 81
Com menos de 100 avaliações: 21 de 81
Com menos de 200 avaliações: 29 de 81

Produtos em alerta: nota de 3.5 a 3.9 | média 3.82 | catálogo (ponderada): 4.16
